# PipeDream - Auditoría Automática de Cambios en Planos de Ingeniería (P&ID)

Este notebook implementa un flujo automatizado para detectar, recortar y auditar visualmente los cambios entre versiones de planos de ingeniería (P&ID). Utiliza técnicas de visión artificial para identificar áreas modificadas, recorta las zonas relevantes y emplea modelos de IA para verificar si las correcciones han sido aplicadas correctamente según las instrucciones de revisión (colores y anotaciones).

**Objetivos principales:**
- Detectar visualmente diferencias entre versiones de planos PDF.
- Recortar automáticamente las zonas de cambio para su análisis.
- Auditar con IA el cumplimiento de las instrucciones de revisión (rojo, amarillo, azul).
- Generar informes estructurados para control de calidad.

Este flujo es útil para equipos de ingeniería, QA y documentación técnica que requieren validar de forma eficiente la correcta implementación de revisiones en planos complejos.

In [1]:
# !pip install PyMuPDF opencv-python numpy openai

In [2]:
import fitz
import cv2
import numpy as np
import os
import base64
import json
import httpx
from openai import AzureOpenAI

In [3]:
def detectar_cambios_visuales(pdf_path, output_path, zoom=2.0, kernel_size=15):
    """
    Parametros:
    pdf_path: Ruta del PDF a procesar.
    output_path: Ruta del PDF de salida con las marcas de cambio.
    zoom: Factor de zoom para mejorar la calidad de la imagen.
    kernel_size: Tamaño del kernel para la dilatación (fusionar elementos cercanos
    Returns:
    None

    Detecta cambios basándose en visión artificial (saturación de color) y fusión morfológica.
    """
    print(f"1. Procesando '{pdf_path}'...")
    doc = fitz.open(pdf_path)
    pagina = doc[0] # Primera página
    bboxes_finales = []

    # --- RENDERIZADO A IMAGEN ---
    # Renderizamos con zoom para tener buena calidad
    matriz = fitz.Matrix(zoom, zoom)
    pix = pagina.get_pixmap(matrix=matriz)
    
    # Convertimos de formato PyMuPDF a formato OpenCV (numpy array)
    img_data = np.frombuffer(pix.samples, dtype=np.uint8)
    img = img_data.reshape(pix.h, pix.w, pix.n)
    
    # Si tiene canal alfa (transparencia), lo quitamos, si no, convertimos RGB a BGR
    if pix.n >= 4:
        img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)
    else:
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

    # --- DETECCIÓN DE COLOR (Saturación) ---
    # Convertimos a espacio HSV (Matiz, Saturación, Valor)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    # Extraemos el canal de Saturación (S)
    # Los grises/negros/blancos tienen S muy baja (cerca de 0).
    # Los colores vivos tienen S alta.
    saturacion = hsv[:, :, 1]
    
    # Filtramos: Nos quedamos con todo lo que tenga saturación > 20 (ajustable)
    _, mask = cv2.threshold(saturacion, 20, 255, cv2.THRESH_BINARY)

    # --- FUSIÓN (Dilatación) ---
    # Aquí ocurre la magia: Expandimos lo blanco para unir islas cercanas
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    mask_dilatada = cv2.dilate(mask, kernel, iterations=2)

    # --- ENCONTRAR CONTORNOS ---
    contornos, _ = cv2.findContours(mask_dilatada, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    print(f"   -> Se han detectado {len(contornos)} áreas de cambio potenciales.")

    # --- DIBUJAR EN EL PDF ORIGINAL ---
    shape = pagina.new_shape()
    contador_cambios = 0

    for cnt in contornos:
        # Obtenemos el rectángulo en píxeles de la imagen
        x, y, w, h = cv2.boundingRect(cnt)
        
        # Filtro de ruido: Si el área es muy pequeña, la ignoramos
        if w * h < 500: 
            continue

        # Convertimos coordenadas de Píxeles (Imagen) a Puntos (PDF)
        # Dividimos por el zoom que aplicamos al principio
        rect_pdf = fitz.Rect(x / zoom, y / zoom, (x + w) / zoom, (y + h) / zoom)
        bboxes_finales.append(rect_pdf)

        # Dibujamos
        shape.draw_rect(rect_pdf)
        shape.insert_text((rect_pdf.x0, rect_pdf.y0 - 5), f"Mod {contador_cambios+1}", color=(0, 1, 0))
        contador_cambios += 1

    shape.finish(color=(0, 1, 0), width=1.5)
    shape.commit()
    
    doc.save(output_path)
    print(f"¡HECHO! Se han marcado {contador_cambios} grupos finales en '{output_path}'")

    return bboxes_finales

In [4]:
def aplicar_margen_seguro(rect, margen, page_width, page_height):
    """
    Expande un rectángulo por un margen dado, asegurándose de no salir
    de los límites de la página.
    """
    nuevo_rect = fitz.Rect(rect)
    nuevo_rect.x0 = max(0, nuevo_rect.x0 - margen)
    nuevo_rect.y0 = max(0, nuevo_rect.y0 - margen)
    nuevo_rect.x1 = min(page_width, nuevo_rect.x1 + margen)
    nuevo_rect.y1 = min(page_height, nuevo_rect.y1 + margen)
    return nuevo_rect

def extraer_secciones_comparativas(path_original, path_modificado, rects, output_folder, zoom=2.0, margen_draft=60):
    """
    Recorta las secciones correspondientes en ambos PDFs.
    
    Args:
        path_original: Ruta del PDF con anotaciones (Draft).
        path_modificado: Ruta del PDF final (Master).
        rects: Lista de objetos fitz.Rect obtenidos en el paso anterior.
        output_folder: Carpeta donde guardar las imágenes recortadas.
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    doc_orig = fitz.open(path_original)
    doc_mod = fitz.open(path_modificado)
    
    # Asumimos que estamos comparando la página 0 con la 0 (ajustar si es multipágina)
    page_orig = doc_orig[0]
    page_mod = doc_mod[0]
    
    matriz = fitz.Matrix(zoom, zoom) # Para alta resolución en el recorte

    pares_rutas = []

    for i, rect in enumerate(rects):
        # --- RECORTE (CLIPPING) ---
        # El método get_pixmap acepta un argumento 'clip' que es el rectángulo en puntos
        pix_orig = page_orig.get_pixmap(matrix=matriz, clip=rect)
        pix_mod = page_mod.get_pixmap(matrix=matriz, clip=aplicar_margen_seguro(rect, margen_draft, page_mod.rect.width, page_mod.rect.height))
        
        # Rutas de salida
        filename_orig = f"{output_folder}/cambio_{i}_original.png"
        filename_mod = f"{output_folder}/cambio_{i}_modificado.png"
        
        # Guardar imágenes
        pix_orig.save(filename_orig)
        pix_mod.save(filename_mod)
        
        pares_rutas.append((filename_orig, filename_mod))

    return pares_rutas

In [5]:
# Función de debug para comprobar las bounding boxes de todos los documentos
def test_strat_all():
    for i in range(6, 56):
        if i < 10:
            try:
                detectar_cambios_visuales(
                    MASTER_PATH + f"0{i}.pdf",
                    "/Workspace/Users/miguel.vera@alumnos.upm.es/ARCHIVOS/RESULTS/debug_bboxes_" + f"0{i}.pdf"
                )
            except:
                pass
        else:
            try:
                detectar_cambios_visuales(
                    MASTER_PATH + f"{i}.pdf",
                    "/Workspace/Users/miguel.vera@alumnos.upm.es/ARCHIVOS/RESULTS/debug_bboxes_" + f"{i}.pdf"
                )
            except:
                pass

In [6]:
AZURE_ENDPOINT = "https://dev-xton-cross-oai.openai.azure.com/" 
AZURE_API_KEY = """7ghSydRepKlFCzRangWjaT9E88mOl7S3CoFQ6MOTxmiiMyTO0SipJQQJ99BKACfhMk5XJ3w3AAABACOGRr3K""" 
API_VERSION = "2024-02-15-preview" 
DEPLOYMENT_NAME = "gpt-4o" 

# --- CONFIGURACIÓN DE RED (SSL BYPASS) ---
cliente_inseguro = httpx.Client(verify=False)

client = AzureOpenAI(
    azure_endpoint=AZURE_ENDPOINT,
    api_key=AZURE_API_KEY,
    api_version=API_VERSION,
    http_client=cliente_inseguro
)

def codificar_imagen(ruta_imagen):
    """Codifica imagen a Base64."""
    with open(ruta_imagen, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def auditar_cambio_visual(ruta_draft, ruta_final):
    print(f"Comparando:\n   A: {ruta_draft}\n   B: {ruta_final} ...")
    
    try:
        img_draft_b64 = codificar_imagen(ruta_draft)
        img_final_b64 = codificar_imagen(ruta_final)
    except FileNotFoundError as e:
        return {"veredicto": "ERROR", "explicacion": f"No encuentro el archivo: {e}"}

    # --- EL PROMPT DEL AUDITOR ---
    prompt_auditor = """
    Actúa como un Ingeniero Senior de Control de Calidad (QA) especializado en planos de ingeniería (P&ID). 
    Tu tarea es auditar rigurosamente si las correcciones marcadas en el plano "DRAFT" se han ejecutado correctamente en el plano "FINAL".

    INPUT:
    Recibirás dos imágenes recortadas de la misma zona del plano:
    1. IMAGEN DRAFT: Contiene las marcas de revisión en colores (Rojo, Amarillo, Azul).
    2. IMAGEN FINAL: Contiene el resultado limpio (en blanco y negro).

    Ten en cuenta que las elipses en azul con un texto dentro como: C0... o GEN... son instrucciones técnicas por lo que no las tengas en cuenta, por lo que no incluyas el color azul si la elipse es la única azul que hay.

    REGLAS DE LÓGICA Y COLORES (Prioridad Absoluta):

    Debes identificar qué combinación de colores aparece en el DRAFT y aplicar SU regla específica:

    CASO 1: SOLO AMARILLO (Eliminación Simple)
    - Intención: Borrar el objeto existente.
    - Verificación: Comprueba que el objeto resaltado/tachado en amarillo en el DRAFT ha desaparecido completamente en el FINAL.

    CASO 2: SOLO ROJO (Adición Simple)
    - Intención: Añadir un nuevo objeto.
    - Verificación: Comprueba que lo dibujado en rojo en el DRAFT aparece dibujado en tinta NEGRA en el FINAL.

    CASO 3: SOLO AZUL (Instrucción/Metadatos)
    - Intención: Ejecutar una orden técnica (ej: "Delete Tracing", "Ver Detalle").
    - Verificación: Lee la instrucción azul y verifica visualmente si el FINAL cumple esa condición lógica.

    CASO 4: ROJO + AMARILLO (Sustitución)
    - Intención: Reemplazar lo viejo por lo nuevo.
    - Verificación: Debes cumplir DOS condiciones:
      1. El objeto amarillo debe haber desaparecido.
      2. El objeto rojo debe aparecer (en negro) en el FINAL, reemplazando lo amarillo.

    CASO 5: ROJO + AZUL (Adición Guiada)
    - Intención: Añadir un objeto siguiendo una directriz espacial o lógica específica.
    - Verificación: Comprueba que el objeto rojo aparece en el FINAL (en negro) Y que su posición coincide con la indicación azul.

    CASO 6: ROJO + AMARILLO + AZUL (Sustitución Compleja Guiada)
    - Intención: Borrar lo viejo y colocar lo nuevo siguiendo instrucciones precisas.
    - Verificación: Debes cumplir TRES condiciones:
      1. El objeto amarillo ha sido eliminado.
      2. El objeto rojo aparece en el FINAL (en negro).
      3. La colocación cumple la instrucción azul.

    SALIDA REQUERIDA (Formato JSON Estricto):
    Devuelve SOLO este JSON:
    {
      "colores_detectados": ["Rojo", "Amarillo", "Azul"], 
      "Tipo de instrucción": ["TEXT_MOD", "SYMBOL_ADD", "SYMBOL_DEL", "INSTR_TRACING", "COMPLEX_REPLACE"],
      "caso_identificado": "CASO X (Nombre del caso)",
      "veredicto": "APROBADO" o "FALLIDO",
      "confianza": 0-100,
      "explicacion": "Breve justificación técnica."
    }
    """

    try:
        response = client.chat.completions.create(
            model=DEPLOYMENT_NAME,
            messages=[
                {"role": "system", "content": "Eres un auditor de planos estricto. Responde solo en JSON."},
                {"role": "user", "content": [
                    {"type": "text", "text": prompt_auditor},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_draft_b64}"}},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_final_b64}"}}
                ]}
            ],
            temperature=0.0, 
            max_completion_tokens=300,
            response_format={"type": "json_object"}
        )
        
        # Limpieza y parseo del JSON
        texto = response.choices[0].message.content
        texto_limpio = texto.replace("```json", "").replace("```", "").strip()
        return json.loads(texto_limpio)

    except Exception as e:
        print(f"Error en la auditoría: {e}")
        return {"veredicto": "ERROR", "explicacion": str(e)}

In [7]:
MASTER_PATH = "MASTER/80080-ED-PRC-0100-PID-0"
DRAFT_PATH = "DRAFT/80080-ED-PRC-0100-PID-0"
num = "06"
rects = detectar_cambios_visuales(
    MASTER_PATH + f"{str(num)}.pdf",
    "debug_bboxes.pdf")
extraer_secciones_comparativas(
    MASTER_PATH + f"{str(num)}.pdf",
    DRAFT_PATH + f"{str(num)}.pdf",
    rects,
    f"/Workspace/Users/miguel.vera@alumnos.upm.es/ARCHIVOS/RESULTS/{str(num)}")

1. Procesando 'MASTER/80080-ED-PRC-0100-PID-006.pdf'...
   -> Se han detectado 31 áreas de cambio potenciales.
¡HECHO! Se han marcado 31 grupos finales en 'debug_bboxes.pdf'


[('/Workspace/Users/miguel.vera@alumnos.upm.es/ARCHIVOS/RESULTS/06/cambio_0_original.png',
  '/Workspace/Users/miguel.vera@alumnos.upm.es/ARCHIVOS/RESULTS/06/cambio_0_modificado.png'),
 ('/Workspace/Users/miguel.vera@alumnos.upm.es/ARCHIVOS/RESULTS/06/cambio_1_original.png',
  '/Workspace/Users/miguel.vera@alumnos.upm.es/ARCHIVOS/RESULTS/06/cambio_1_modificado.png'),
 ('/Workspace/Users/miguel.vera@alumnos.upm.es/ARCHIVOS/RESULTS/06/cambio_2_original.png',
  '/Workspace/Users/miguel.vera@alumnos.upm.es/ARCHIVOS/RESULTS/06/cambio_2_modificado.png'),
 ('/Workspace/Users/miguel.vera@alumnos.upm.es/ARCHIVOS/RESULTS/06/cambio_3_original.png',
  '/Workspace/Users/miguel.vera@alumnos.upm.es/ARCHIVOS/RESULTS/06/cambio_3_modificado.png'),
 ('/Workspace/Users/miguel.vera@alumnos.upm.es/ARCHIVOS/RESULTS/06/cambio_4_original.png',
  '/Workspace/Users/miguel.vera@alumnos.upm.es/ARCHIVOS/RESULTS/06/cambio_4_modificado.png'),
 ('/Workspace/Users/miguel.vera@alumnos.upm.es/ARCHIVOS/RESULTS/06/cambio_5

In [8]:
ruta_original = "RESULTS/06/06/cambio_16_original.png"
ruta_modificado = "RESULTS/06/06/cambio_16_modificado.png"

if os.path.exists(ruta_original) and os.path.exists(ruta_modificado):
    resultado = auditar_cambio_visual(ruta_original, ruta_modificado)
    print("\n--- INFORME DE AUDITORÍA ---")
    print(json.dumps(resultado, indent=2, ensure_ascii=False))
else:
    print(f"Faltan imágenes de prueba en:\n{ruta_original}")

Comparando:
   A: RESULTS/06/06/cambio_16_original.png
   B: RESULTS/06/06/cambio_16_modificado.png ...

--- INFORME DE AUDITORÍA ---
{
  "colores_detectados": [
    "Rojo",
    "Azul"
  ],
  "Tipo de instrucción": [
    "SYMBOL_ADD",
    "INSTR_TRACING"
  ],
  "caso_identificado": "CASO 5 (Adición Guiada)",
  "veredicto": "APROBADO",
  "confianza": 95,
  "explicacion": "El objeto rojo 'NOTE 6' aparece en el FINAL en tinta negra y su posición coincide con la indicación azul 'C010'."
}


## Conclusión

Este notebook ha demostrado un flujo automatizado y eficiente para la auditoría visual de cambios en planos de ingeniería (P&ID). A través de técnicas de visión artificial y modelos de IA, se han detectado, recortado y verificado las modificaciones entre versiones de planos, asegurando el cumplimiento de las instrucciones de revisión. Este proceso facilita el control de calidad, reduce errores humanos y agiliza la validación documental en proyectos de ingeniería complejos.

**Siguientes pasos realizados:**
- Integración de las funciones implementadas en este notebook en una REST API.
- Visualizar resultados en una aplicación web full-stack (se ha realizado con front-end en React y back-end en FastAPI)

Gracias por utilizar este notebook para mejorar la calidad y eficiencia en la gestión de revisiones de planos de ingeniería.